In [1]:
import os, sys, glob, shutil
import pandas as pd

print("Python:", sys.version)
print("CWD:", os.getcwd())

print("configs exists:", os.path.isdir("configs"))
print("results exists:", os.path.isdir("results"))
if not os.path.isdir("results"):
    os.makedirs("results", exist_ok=True)

print("configs/*.json:", glob.glob("configs/*.json")[:10])


Python: 3.10.15 | packaged by Anaconda, Inc. | (main, Oct  3 2024, 07:22:19) [MSC v.1929 64 bit (AMD64)]
CWD: c:\research\sustaibabilityplot\CLEAR_ATS\CLEAR_ATS
configs exists: True
results exists: True
configs/*.json: ['configs\\california.json', 'configs\\ohio.json', 'configs\\us_average.json']


In [2]:
import subprocess, sys

cmd = [
    sys.executable, "run.py",
    "--no-app",
    "--scenarios", "us_average",
    "--years", "68",
    "--policy", "baseline",
    "--mc", "0",
    "--seed", "0",
]
print("Running:", " ".join(cmd))
subprocess.check_call(cmd)


Running: c:\Users\17264\miniconda3\envs\ucmctrack_env\python.exe run.py --no-app --scenarios us_average --years 68 --policy baseline --mc 0 --seed 0


0

In [3]:
import subprocess, sys

cmd = [
    sys.executable, "run.py",
    "--no-app",
    "--scenarios", "us_average",
    "--years", "68",
    "--policy", "baseline",
    "--mc", "200",     # must be > 1 to get quantiles
    "--seed", "42",
]
print("Running:", " ".join(cmd))
subprocess.check_call(cmd)


Running: c:\Users\17264\miniconda3\envs\ucmctrack_env\python.exe run.py --no-app --scenarios us_average --years 68 --policy baseline --mc 200 --seed 42


0

In [4]:
import glob, os, shutil

scenario = "us_average"

candidates = sorted(glob.glob(os.path.join("results", f"{scenario}*quantiles.csv")))
print("Quantiles candidates:")
for p in candidates:
    print("  ", p)

ui_path = os.path.join("results", f"{scenario}_quantiles.csv")

if len(candidates) == 0:
    raise FileNotFoundError("No quantiles csv found. Confirm you ran with --mc > 1 and results/ is correct.")

# Prefer the baseline policy/model file if it exists
preferred = [p for p in candidates if "__policy-baseline__model-" in os.path.basename(p)]
src = preferred[0] if len(preferred) > 0 else candidates[0]

shutil.copyfile(src, ui_path)
print("Copied:", src)
print("To UI name:", ui_path)


Quantiles candidates:
   results\us_average__policy-baseline__model-fixed_table_metrics_quantiles.csv
   results\us_average__policy-baseline__model-fixed_table_quantiles.csv
   results\us_average_quantiles.csv
Copied: results\us_average__policy-baseline__model-fixed_table_metrics_quantiles.csv
To UI name: results\us_average_quantiles.csv


In [5]:
import pandas as pd
import os

qpath = os.path.join("results", "us_average_quantiles.csv")
qdf = pd.read_csv(qpath)

# Example variable: ATS Emissions (kg CO2)
col_p05 = "ATS Emissions (kg CO2)_p05"
col_p50 = "ATS Emissions (kg CO2)_p50"
col_p95 = "ATS Emissions (kg CO2)_p95"

missing = [c for c in [col_p05, col_p50, col_p95] if c not in qdf.columns]
if len(missing) > 0:
    print("Missing columns:", missing)
    print("Available columns (first 30):", list(qdf.columns)[:30])
else:
    out = qdf[["Year", col_p05, col_p50, col_p95]].head(15)
    print(out.to_string(index=False))
    band_width = (qdf[col_p95] - qdf[col_p05]).describe()
    print("\nBand width summary (p95 - p05):")
    print(band_width.to_string())


Missing columns: ['ATS Emissions (kg CO2)_p05', 'ATS Emissions (kg CO2)_p50', 'ATS Emissions (kg CO2)_p95']
Available columns (first 30): ['metric', 'quantile', 'value']
